# 03 — Train Orthoptera Classifier
Fine-tune a CNN on the prepared label splits using OpenSoundscape, then export the trained model to `models/` for use in BASE.

**Kernel:** `Python (orthoptera-training)`  
**Prerequisites:** Run `02_prepare_labels.ipynb` first to generate `training/data/train.csv` etc.

In [ ]:
import torch
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
import opensoundscape.ml.cnn
import opensoundscape.utils
from opensoundscape import CNN, SpectrogramPreprocessor
import wandb
import random
import numpy as np

# Set all seeds for reproducibility.
opensoundscape.utils.set_seed(18, verbose=False)

PROJECT_ROOT = Path.cwd().parent.parent
DATA_DIR   = PROJECT_ROOT / "training" / "data"
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv", index_col=[0, 1, 2])
val   = pd.read_csv(DATA_DIR / "val.csv",   index_col=[0, 1, 2])
test  = pd.read_csv(DATA_DIR / "test.csv",  index_col=[0, 1, 2])

print(f"Classes: {list(train.columns)}")
print(f"Train: {len(train)}  Val: {len(val)}  Test: {len(test)}")


In [ ]:
# ── View available model architectures ─────────────────────────────────────────────────
print(opensoundscape.ml.cnn_architectures.list_architectures())


In [ ]:
# ── Build model ──────────────────────────────────────────────────────────────
model = CNN(
    architecture="resnet18",
    classes=list(train.columns),
    sample_duration=4.0
)
print("Model built — architecture: resnet18\n")
print(f"Classes: {model.classes}\n")

# Utilise GPU and verify.
model.network.to("mps")
print(next(model.network.parameters()).device)

print(model.network)


In [ ]:
# import torch.nn as nn

# model.network.fc = nn.Sequential(
#     nn.Dropout(p=0.5),
#     nn.Linear(512, len(model.classes))
# )
print(f"Model Hyperparameters: {model.optimizer_params}\n")
# print(f"Model FC Layer:\n{model.network.fc}")

In [ ]:
# ── Add/tweak preprocessor augmentations and verify steps and parameters ─────────────────────────
# 140kHz reflects the highest recorded frequency in EcoSoundSet.
model.preprocessor.pipeline.bandpass.set(max_f=140000, out_of_bounds_ok=True)
model.preprocessor.pipeline.random_trim_audio.set(random_trim=False)
# Standardised 280 kHz sample rate ensures all resulting tensors have the same frequency scale.
model.preprocessor.pipeline.load_audio.set(sample_rate=280000)
# Silence noisy warnings about clip lengths being .0000000001s (or similar) too short.
model.preprocessor.pipeline.load_audio.set(out_of_bounds_mode="ignore")

print(model.preprocessor.pipeline)
for name, step in model.preprocessor.pipeline.items():
    print(f"{name}: \n{step.params}\n")


In [ ]:
print(model.loss_fn)

In [ ]:
# ── View augmentation effects on resulting tensor images. ─────────────────────────
# Individual augmentations can be disabled (set vals to True) here to pinpoint specific effect.
model.preprocessor.pipeline.time_mask.bypass = False
model.preprocessor.pipeline.frequency_mask.bypass = False
model.preprocessor.pipeline.random_trim_audio.bypass = False
model.preprocessor.pipeline.add_noise.bypass = False
model.preprocessor.pipeline.random_affine.bypass = False

from opensoundscape.preprocess.utils import show_tensor_grid
from opensoundscape.ml.datasets import AudioFileDataset

# Create AudioFileDataSet instance using training dataframe and the model's preprocessor.
dataset = AudioFileDataset(train, model.preprocessor)

sample = dataset[867].data
print(sample.max(), sample.min(), sample.mean())

# Extract the data tensors for 9 selected samples.
tensors = [dataset[i].data for i in range(867, 876)]

# Extract the active text labels (where value > 0) for each of those 9 samples.
sample_labels = [list(dataset[i].labels[dataset[i].labels > 0].index) for i in range(867, 876)]

# Plot the grid (3 columns wide)
_ = show_tensor_grid(tensors, 3, labels=sample_labels)

plt.show()


In [ ]:
# ── View initial spectrogram created by preprocessor. ─────────────────────────
pre = SpectrogramPreprocessor(4.0)
pre.pipeline = model.preprocessor.pipeline[0:4]
print(pre.pipeline)

dataset = AudioFileDataset(train, pre)
dataset[290].data.plot() #542


In [ ]:
# ── Ensure no augmentations are bypassed after changes made in previous cells. ────────────────────
model.preprocessor.pipeline.time_mask.bypass = False
model.preprocessor.pipeline.frequency_mask.bypass = False
model.preprocessor.pipeline.random_trim_audio.bypass = False
model.preprocessor.pipeline.add_noise.bypass = False
model.preprocessor.pipeline.random_affine.bypass = False

print(model.preprocessor.pipeline)


In [ ]:
# ── Train ───────────────────────────────────────────────────────────────────
# wandb.login()

# session = wandb.init(
#     entity="dawsonmccall62-personal",
#     project="BASE",
#     name="july_17th_overfitting_experiment",
# )

model.train(
    train,
    val,
    epochs=20,
    batch_size=32,
    save_path=MODELS_DIR / "orthoptera_checkpoints",
    save_interval=1,
    num_workers=2,
    # wandb_session=session
)

#session.finish()

In [ ]:
# ── Acquire per-epoch mAP metrics for plotting ──────────────────────────────
train_mAP = []
for key in model.train_metrics.keys():
    train_mAP.append(round(model.train_metrics[key]["map"], 3))

val_mAP = []
for key in model.valid_metrics.keys():
    val_mAP.append(round(model.valid_metrics[key]["map"], 3))


print(f"Train mAP: {train_mAP}\n\nValidation mAP: {val_mAP}")

In [ ]:
# ── Evaluate on held-out test set (3,137 clips) ──────────────────────────────
from sklearn.metrics import classification_report

scores = model.predict(test)
y_true = test.values
y_pred = (scores.values > 0.4).astype(int)
print(classification_report(
    y_true, y_pred,
    target_names=list(train.columns),
    zero_division=0,
))


In [ ]:
# ── Evaluate best model on held-out test set (3,137 clips) ──────────────────────────────
from sklearn.metrics import classification_report
from opensoundscape.ml.cnn import load_model

best_model = torch.load(MODELS_DIR / "orthoptera_checkpoints" / "best.model", weights_only=False)
#best_model = torch.load(MODELS_DIR / "orthoptera_checkpoints" / "August" / "orthoptera_uk_v2_15.model", weights_only=False)
print(best_model.current_epoch)
scores = best_model.predict(test)
y_true = test.values
y_pred = (scores.values > 0.3).astype(int)
print(classification_report(
    y_true, y_pred,
    target_names=list(train.columns),
    zero_division=0,
))


In [ ]:
model = torch.load(MODELS_DIR / "orthoptera_checkpoints" / "epoch-14.model", weights_only=False)
print(model.train_metrics[0], model.valid_metrics[0])

In [ ]:
# ── Threshold sweep — find F1-maximising threshold per class ──────────────────
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = np.arange(0.1, 0.91, 0.05)
class_names = y_true.columns if hasattr(y_true, 'columns') else range(y_true.shape[1])

results = []

for i, class_name in enumerate(class_names):
    y_true_class = y_true.iloc[:, i] if hasattr(y_true, 'iloc') else y_true[:, i]
    scores_class = scores.iloc[:, i] if hasattr(scores, 'iloc') else scores[:, i]

    best_f1 = -1
    best_t = 0.4
    for t in thresholds:
        y_pred_class = (scores_class > t).astype(int)
        f1 = f1_score(y_true_class, y_pred_class, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    precision = precision_score(y_true_class, (scores_class > best_t).astype(int), zero_division=0)
    recall = recall_score(y_true_class, (scores_class > best_t).astype(int), zero_division=0)
    
    results.append({
        'class': class_name,
        'best_threshold': round(best_t, 2),
        'f1': round(best_f1, 3),
        'precision': round(precision, 3),
        'recall': round(recall, 3)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))


In [ ]:
# ── Save final model ──────────────────────────────────────────────────────────
# OpenSoundscape saves as a .model file (a torch pickle).
# This is the file you point insect.py at in BASE.

model_path = MODELS_DIR / "orthoptera_uk.model"
model.save(model_path)
print(f"Model saved: {model_path}")
print()
print("Next step: set 'model_path' in config/settings.yaml under 'insect:'")
print("and activate 'insect' in classifiers.active")

### Results — 5-epoch ResNet18 (CPU, May 2026)

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.96 | 0.61 | **0.75** | 2,530 |
| *Chorthippus brunneus brunneus* | 0.81 | 0.23 | **0.36** | 73 |
| *Pholidoptera griseoaptera* | 0.76 | 0.12 | **0.21** | 1,393 |
| *Leptophyes punctatissima* | 0.85 | 0.08 | **0.15** | 359 |
| *Gryllus campestris* | 0.94 | 0.06 | **0.12** | 929 |
| *Roeseliana roeselii* | 0.59 | 0.06 | **0.10** | 741 |
| *Pseudochorthippus parallelus* | 0.77 | 0.03 | **0.05** | 358 |
| *Omocestus viridulus* | 1.00 | 0.01 | **0.03** | 76 |

**Macro F1: 0.22 — precision is high (low false-positive rate) but recall is low.**

Pattern: the model is conservative after only 5 epochs — it fires confidently when it fires,
but misses most detections. *Tettigonia* performs well (most training data: 6,270 clips).

**To improve:**
- More epochs (20–30) — the main lever
- Lower `min_confidence` threshold in `config/settings.yaml` (try 0.3)
- GPU training would make 30 epochs feasible in <1 hour

**Threshold sweep verdict:** macro-F1 is flat across all thresholds (0.215–0.225). Lowering the threshold does not help — the model's score distribution is too compressed after only 5 epochs. More training epochs is the only meaningful lever.


### Results — 5-epoch ResNet18 (GPU, #1)
### Alteration: fixes for Multi-hot Encoding and proper start and end times

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.83 | 0.62 | **0.71** | 887 |
| *Chorthippus brunneus brunneus* | 0.93 | 0.34 | **0.50** | 41 |
| *Pholidoptera griseoaptera* | 0.99 | 0.41 | **0.58** | 509 |
| *Leptophyes punctatissima* | 0.61 | 0.15 | **0.24** | 238 |
| *Gryllus campestris* | 0.98 | 0.91 | **0.94** | 861 |
| *Roeseliana roeselii* | 0.51 | 0.50 | **0.50** | 693 |
| *Pseudochorthippus parallelus* | 0.89 | 0.37 | **0.53** | 295 |
| *Omocestus viridulus* | 0.97 | 0.55 | **0.70** | 64 |

**Macro F1: 0.59.**

### Results — 5-epoch ResNet18 (GPU, #2)
### Alterations: splitting InsectSet459 into 4 second clips like EcoSoundSet

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.85 | 0.76 | **0.80** | 887 |
| *Chorthippus brunneus brunneus* | 0.64 | 0.44 | **0.52** | 41 |
| *Pholidoptera griseoaptera* | 0.84 | 0.49 | **0.62** | 509 |
| *Leptophyes punctatissima* | 0.50 | 0.03 | **0.06** | 238 |
| *Gryllus campestris* | 0.98 | 0.92 | **0.95** | 861 |
| *Roeseliana roeselii* | 0.73 | 0.24 | **0.36** | 693 |
| *Pseudochorthippus parallelus* | 0.75 | 0.31 | **0.44** | 295 |
| *Omocestus viridulus* | 0.95 | 0.66 | **0.78** | 64 |

**Macro F1: 0.56**

### Results — 5-epoch ResNet18 (GPU, #3)
### Alterations: taking the 4s window with the greatest energy above 2 kHz frequency for each InsectSet459 clip.

Many InsectSet clips are poor quality or mostly silence, so instead of using the first 4s or splitting the whole clip into 4s segments we target higher frequencies (> 2000 Hz) and take the 4s segment with the most noise in that range. As the metrics show, this change slightly improved overall F1 Macro:

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.85 | 0.76 | **0.80** | 887 |
| *Chorthippus brunneus brunneus* | 0.64 | 0.44 | **0.52** | 41 |
| *Pholidoptera griseoaptera* | 0.84 | 0.49 | **0.62** | 509 |
| *Leptophyes punctatissima* | 0.50 | 0.03 | **0.06** | 238 |
| *Gryllus campestris* | 0.98 | 0.92 | **0.95** | 861 |
| *Roeseliana roeselii* | 0.73 | 0.24 | **0.36** | 693 |
| *Pseudochorthippus parallelus* | 0.75 | 0.31 | **0.44** | 295 |
| *Omocestus viridulus* | 0.95 | 0.66 | **0.78** | 64 |
| **Micro avg** | 0.85 | 0.54 | **0.66** | 3588 |
| **Macro avg** | 0.83 | 0.49 | **0.60** | 3588 |
| **Weighted avg** | 0.83 | 0.54 | **0.63** | 3588 |
| **Samples avg** | 0.60 | 0.57 | **0.58** | 3588 |

### Results — 5-epoch ResNet34 (GPU, #4)
### Alterations: first training attempt with the ResNet34 architecture.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.83 | 0.50 | **0.62** | 887 |
| *Chorthippus brunneus brunneus* | 1.00 | 0.54 | **0.70** | 41 |
| *Pholidoptera griseoaptera* | 0.97 | 0.50 | **0.65** | 509 |
| *Leptophyes punctatissima* | 0.30 | 0.08 | **0.13** | 238 |
| *Gryllus campestris* | 0.97 | 0.94 | **0.96** | 861 |
| *Roeseliana roeselii* | 0.54 | 0.56 | **0.55** | 693 |
| *Pseudochorthippus parallelus* | 0.82 | 0.37 | **0.51** | 295 |
| *Omocestus viridulus* | 0.94 | 0.70 | **0.80** | 64 |
| **Micro avg** | 0.80 | 0.58 | **0.68** | 3588 |
| **Macro avg** | 0.80 | 0.52 | **0.62** | 3588 |
| **Weighted avg** | 0.80 | 0.58 | **0.66** | 3588 |
| **Samples avg** | 0.64 | 0.62 | **0.63** | 3588 |

### Results — 5-epoch ResNet34 (GPU, #5)
### Alterations: Include negative clips (clips with no Orthoptera annotations whatsoever) to help model ignore anthropogenic and non-target biotic sounds in target Orthoptera clips.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.82 | 0.64 | **0.72** | 887 |
| *Chorthippus brunneus brunneus* | 0.89 | 0.20 | **0.32** | 41 |
| *Pholidoptera griseoaptera* | 0.87 | 0.54 | **0.66** | 509 |
| *Leptophyes punctatissima* | 0.79 | 0.05 | **0.09** | 238 |
| *Gryllus campestris* | 0.96 | 0.84 | **0.89** | 861 |
| *Roeseliana roeselii* | 0.51 | 0.61 | **0.55** | 693 |
| *Pseudochorthippus parallelus* | 0.55 | 0.43 | **0.48** | 295 |
| *Omocestus viridulus* | 0.69 | 0.38 | **0.48** | 64 |
| **Micro avg** | 0.75 | 0.60 | **0.67** | 3588 |
| **Macro avg** | 0.76 | 0.46 | **0.53** | 3588 |
| **Weighted avg** | 0.77 | 0.60 | **0.65** | 3588 |
| **Samples avg** | 0.45 | 0.43 | **0.43** | 3588 |

### Results — 5-epoch ResNet18 (GPU, #6)
### Alterations: Fix Opensoundscape bandpass augmentation which arbitrarily cuts recording data off at 11025hz (far too low for multiple target species).

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.88 | 0.74 | **0.80** | 887 |
| *Chorthippus brunneus brunneus* | 0.88 | 0.17 | **0.29** | 41 |
| *Pholidoptera griseoaptera* | 0.95 | 0.68 | **0.79** | 509 |
| *Leptophyes punctatissima* | 0.69 | 0.93 | **0.79** | 238 |
| *Gryllus campestris* | 0.95 | 0.93 | **0.94** | 861 |
| *Roeseliana roeselii* | 0.88 | 0.66 | **0.75** | 693 |
| *Pseudochorthippus parallelus* | 0.70 | 0.68 | **0.69** | 295 |
| *Omocestus viridulus* | 0.95 | 0.58 | **0.73** | 64 |
| **Micro avg** | 0.87 | 0.76 | **0.81** | 3588 |
| **Macro avg** | 0.86 | 0.67 | **0.72** | 3588 |
| **Weighted avg** | 0.88 | 0.76 | **0.81** | 3588 |
| **Samples avg** | 0.83 | 0.80 | **0.80** | 3588 |


### Results — 5-epoch ResNet18 (GPU, #7)
### Alterations: Remove Opensoundscape `random_trim_audio` that neglected the work done to derive the best 4s window in the InsectSet459 clips.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.87 | 0.78 | **0.82** | 887 |
| *Chorthippus brunneus brunneus* | 0.82 | 0.34 | **0.48** | 41 |
| *Pholidoptera griseoaptera* | 0.92 | 0.67 | **0.78** | 509 |
| *Leptophyes punctatissima* | 0.84 | 0.87 | **0.86** | 238 |
| *Gryllus campestris* | 0.95 | 0.93 | **0.94** | 861 |
| *Roeseliana roeselii* | 0.88 | 0.64 | **0.74** | 693 |
| *Pseudochorthippus parallelus* | 0.88 | 0.56 | **0.69** | 295 |
| *Omocestus viridulus* | 0.85 | 0.64 | **0.73** | 64 |
| **Micro avg** | 0.90 | 0.75 | **0.82** | 3588 |
| **Macro avg** | 0.88 | 0.68 | **0.75** | 3588 |
| **Weighted avg** | 0.90 | 0.75 | **0.81** | 3588 |
| **Samples avg** | 0.82 | 0.79 | **0.80** | 3588 |


### Results — 5-epoch ResNet18 (GPU, #8)
### Alterations: Resample classes to ensure each class has equal representation in training.

 Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.89 | 0.80 | **0.84** | 887 |
| *Chorthippus brunneus brunneus* | 0.78 | 0.44 | **0.56** | 41 |
| *Pholidoptera griseoaptera* | 0.94 | 0.78 | **0.85** | 509 |
| *Leptophyes punctatissima* | 0.72 | 0.95 | **0.82** | 238 |
| *Gryllus campestris* | 1.00 | 0.93 | **0.96** | 861 |
| *Roeseliana roeselii* | 0.88 | 0.84 | **0.86** | 693 |
| *Pseudochorthippus parallelus* | 0.79 | 0.74 | **0.76** | 295 |
| *Omocestus viridulus* | 0.96 | 0.72 | **0.82** | 64 |
| **Micro avg** | 0.89 | 0.84 | **0.86** | 3588 |
| **Macro avg** | 0.87 | 0.78 | **0.81** | 3588 |
| **Weighted avg** | 0.90 | 0.84 | **0.86** | 3588 |
| **Samples avg** | 0.89 | 0.87 | **0.86** | 3588 |


### Results — 5-epoch ResNet18 (GPU, #9)
### Alterations: Add negative clips to validation and testing to ensure model is evaluated on realistic conditions.
###
### Performance with previous model (no negative clips included in training).
| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Tettigonia viridissima | 0.68 | 0.80 | 0.73 | 887 |
| Chorthippus brunneus brunneus | 0.32 | 0.44 | 0.37 | 41 |
| Pholidoptera griseoaptera | 0.74 | 0.78 | 0.76 | 509 |
| Leptophyes punctatissima | 0.64 | 0.95 | 0.77 | 238 |
| Gryllus campestris | 0.69 | 0.93 | 0.79 | 861 |
| Roeseliana roeselii | 0.77 | 0.84 | 0.80 | 693 |
| Pseudochorthippus parallelus parallelus | 0.50 | 0.74 | 0.60 | 295 |
| Omocestus viridulus | 0.53 | 0.72 | 0.61 | 64 |
| **Micro avg** | 0.68 | 0.84 | **0.75** | 3588 |
| **Macro avg** | 0.61 | 0.78 | **0.68** | 3588 |
| **Weighted avg** | 0.68 | 0.84 | **0.75** | 3588 |
| **Samples avg** | 0.60 | 0.59 | **0.59** | 3588 |

### Performance with current epoch model (negative clips included in training).
| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| Tettigonia viridissima | 0.60 | 0.82 | **0.69** | 887 |
| Chorthippus brunneus brunneus | 0.24 | 0.54 | **0.34** | 41 |
| Pholidoptera griseoaptera | 0.80 | 0.66 | **0.72** | 509 |
| Leptophyes punctatissima | 0.82 | 0.89 | **0.85** | 238 |
| Gryllus campestris | 0.76 | 0.91 | **0.83** | 861 |
| Roeseliana roeselii | 0.85 | 0.69 | **0.76** | 693 |
| Pseudochorthippus parallelus parallelus | 0.48 | 0.78 | **0.59** | 295 |
| Omocestus viridulus | 0.45 | 0.61 | **0.52** | 64 |
| **Micro avg** | 0.68 | 0.79 | **0.73** | 3588 |
| **Macro avg** | 0.63 | 0.74 | **0.66** | 3588 |
| **Weighted avg** | 0.71 | 0.79 | **0.74** | 3588 |
| **Samples avg** | 0.58 | 0.56 | **0.56** | 3588 |


### Results — 5-epoch ResNet18 (GPU, #10)
### Alterations: Rebalance negative clips in training to ensure equal representation to Orthoptera classes.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.64 | 0.82 | **0.72** | 887 |
| *Chorthippus brunneus brunneus* | 0.20 | 0.54 | **0.29** | 41 |
| *Pholidoptera griseoaptera* | 0.84 | 0.77 | **0.80** | 509 |
| *Leptophyes punctatissima* | 0.76 | 0.89 | **0.82** | 238 |
| *Gryllus campestris* | 0.79 | 0.91 | **0.85** | 861 |
| *Roeseliana roeselii* | 0.75 | 0.71 | **0.73** | 693 |
| *Pseudochorthippus parallelus* | 0.59 | 0.67 | **0.63** | 295 |
| *Omocestus viridulus* | 0.38 | 0.62 | **0.47** | 64 |
| **Micro avg** | 0.70 | 0.80 | **0.75** | 3588 |
| **Macro avg** | 0.62 | 0.74 | **0.66** | 3588 |
| **Weighted avg** | 0.72 | 0.80 | **0.75** | 3588 |
| **Samples avg** | 0.58 | 0.57 | **0.57** | 3588 |


### Results — 30-epoch ResNet18 (GPU, #11)
### Alterations: Retrain with balanced negative clips over 30 epochs to see if epoch count is the trigger.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.68 | 0.84 | **0.75** | 887 |
| *Chorthippus brunneus brunneus* | 0.40 | 0.49 | **0.44** | 41 |
| *Pholidoptera griseoaptera* | 0.85 | 0.77 | **0.81** | 509 |
| *Leptophyes punctatissima* | 0.73 | 0.94 | **0.82** | 238 |
| *Gryllus campestris* | 0.71 | 0.94 | **0.81** | 861 |
| *Roeseliana roeselii* | 0.69 | 0.79 | **0.73** | 693 |
| *Pseudochorthippus parallelus parallelus* | 0.59 | 0.64 | **0.61** | 295 |
| *Omocestus viridulus* | 0.50 | 0.86 | **0.63** | 64 |
| **Micro avg** | 0.70 | 0.83 | **0.76** | 3588 |
| **Macro avg** | 0.64 | 0.78 | **0.70** | 3588 |
| **Weighted avg** | 0.70 | 0.83 | **0.76** | 3588 |
| **Samples avg** | 0.60 | 0.58 | **0.58** | 3588 |

### Associated plot in `04_plot_results` details the overfitting issue highlighted during this model training.

### Results — 30-epoch ResNet18 (GPU, #12)
### Alterations: Train with resample loss and no negative clips.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.91 | 0.73 | **0.81** | 887 |
| *Chorthippus brunneus brunneus* | 0.62 | 0.44 | **0.51** | 41 |
| *Pholidoptera griseoaptera* | 0.99 | 0.69 | **0.81** | 509 |
| *Leptophyes punctatissima* | 0.82 | 0.91 | **0.86** | 238 |
| *Gryllus campestris* | 0.98 | 0.94 | **0.96** | 861 |
| *Roeseliana roeselii* | 0.83 | 0.75 | **0.79** | 693 |
| *Pseudochorthippus parallelus parallelus* | 0.73 | 0.64 | **0.68** | 295 |
| *Omocestus viridulus* | 0.88 | 0.80 | **0.84** | 64 |
| **Micro avg** | 0.90 | 0.78 | **0.83** | 3588 |
| **Macro avg** | 0.84 | 0.74 | **0.78** | 3588 |
| **Weighted avg** | 0.90 | 0.78 | **0.83** | 3588 |
| **Samples avg** | 0.86 | 0.82 | **0.83** | 3588 |


### Results — 20-epoch ResNet18 (GPU, #13)
### Alterations: Train with no resampling (of any kind) and no negative clips.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.89 | 0.74 | **0.81** | 887 |
| *Chorthippus brunneus brunneus* | 0.84 | 0.39 | **0.53** | 41 |
| *Pholidoptera griseoaptera* | 0.91 | 0.81 | **0.86** | 509 |
| *Leptophyes punctatissima* | 0.74 | 0.92 | **0.82** | 238 |
| *Gryllus campestris* | 0.97 | 0.94 | **0.95** | 861 |
| *Roeseliana roeselii* | 0.87 | 0.77 | **0.82** | 693 |
| *Pseudochorthippus parallelus parallelus* | 0.82 | 0.62 | **0.71** | 295 |
| *Omocestus viridulus* | 0.94 | 0.70 | **0.80** | 64 |
| **Micro avg** | 0.89 | 0.80 | **0.84** | 3588 |
| **Macro avg** | 0.87 | 0.74 | **0.79** | 3588 |
| **Weighted avg** | 0.89 | 0.80 | **0.84** | 3588 |
| **Samples avg** | 0.87 | 0.83 | **0.84** | 3588 |


### Results — 20-epoch ResNet18 (GPU, #14)
### Alterations: Train with no resampling (of any kind), no negative clips, and no augmentation.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima*                  | 0.87      | 0.79   | **0.83** | 887     |
| *Chorthippus brunneus brunneus*           | 0.85      | 0.41   | **0.56** | 41      |
| *Pholidoptera griseoaptera*               | 0.92      | 0.83   | **0.87** | 509     |
| *Leptophyes punctatissima*                | 0.87      | 0.94   | **0.90** | 238     |
| *Gryllus campestris*                      | 0.97      | 0.94   | **0.95** | 861     |
| *Roeseliana roeselii*                     | 0.88      | 0.77   | **0.82** | 693     |
| *Pseudochorthippus parallelus parallelus* | 0.88      | 0.66   | **0.76** | 295     |
| *Omocestus viridulus*                     | 0.95      | 0.62   | **0.75** | 64      |
| **Micro avg**                             | 0.91      | 0.82   | **0.86** | 3588    |
| **Macro avg**                             | 0.90      | 0.75   | **0.81** | 3588    |
| **Weighted avg**                          | 0.90      | 0.82   | **0.86** | 3588    |
| **Samples avg**                           | 0.88      | 0.85   | **0.85** | 3588    |


### Results — 20-epoch ResNet18 (GPU, #15)
### Alterations: Retrain with 280 kHz Standardised Sample Rate.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima*                  | 0.85      | 0.82   | **0.84** | 887     |
| *Chorthippus brunneus brunneus*           | 0.88      | 0.34   | **0.49** | 41      |
| *Pholidoptera griseoaptera*               | 0.97      | 0.76   | **0.85** | 509     |
| *Leptophyes punctatissima*                | 0.76      | 0.95   | **0.85** | 238     |
| *Gryllus campestris*                      | 0.99      | 0.87   | **0.93** | 861     |
| *Roeseliana roeselii*                     | 0.85      | 0.82   | **0.84** | 693     |
| *Pseudochorthippus parallelus parallelus* | 0.84      | 0.69   | **0.76** | 295     |
| *Omocestus viridulus*                     | 0.96      | 0.72   | **0.82** | 64      |
| **Micro avg**                             | 0.89      | 0.82   | **0.85** | 3588    |
| **Macro avg**                             | 0.89      | 0.75   | **0.80** | 3588    |
| **Weighted avg**                          | 0.90      | 0.82   | **0.85** | 3588    |
| **Samples avg**                           | 0.88      | 0.85   | **0.85** | 3588    |                            


### Results — 20-epoch ResNet18 (GPU, #16)
### Alterations: Increase weight decay to .001.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima*                  | 0.88      | 0.83   | **0.85** | 887     |
| *Chorthippus brunneus brunneus*           | 0.85      | 0.27   | **0.41** | 41      |
| *Pholidoptera griseoaptera*               | 0.99      | 0.68   | **0.80** | 509     |
| *Leptophyes punctatissima*                | 0.86      | 0.91   | **0.88** | 238     |
| *Gryllus campestris*                      | 0.97      | 0.89   | **0.93** | 861     |
| *Roeseliana roeselii*                     | 0.81      | 0.85   | **0.83** | 693     |
| *Pseudochorthippus parallelus parallelus* | 0.83      | 0.54   | **0.65** | 295     |
| *Omocestus viridulus*                     | 1.00      | 0.56   | **0.72** | 64      |
| **Micro avg**                             | 0.89      | 0.80   | **0.84** | 3588    |
| **Macro avg**                             | 0.90      | 0.69   | **0.76** | 3588    |
| **Weighted avg**                          | 0.90      | 0.80   | **0.84** | 3588    |
| **Samples avg**                           | 0.87      | 0.83   | **0.84** | 35

### Results — 20-epoch ResNet18 (GPU, #17)
### Alterations: Increase weight decay to .005.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima*                  | 0.94      | 0.69   | **0.80** | 887     |
| *Chorthippus brunneus brunneus*           | 0.00      | 0.00   | **0.00** | 41      |
| *Pholidoptera griseoaptera*               | 0.97      | 0.60   | **0.74** | 509     |
| *Leptophyes punctatissima*                | 0.91      | 0.82   | **0.86** | 238     |
| *Gryllus campestris*                      | 0.97      | 0.90   | **0.93** | 861     |
| *Roeseliana roeselii*                     | 0.81      | 0.77   | **0.79** | 693     |
| *Pseudochorthippus parallelus parallelus* | 0.79      | 0.58   | **0.67** | 295     |
| *Omocestus viridulus*                     | 1.00      | 0.42   | **0.59** | 64      |
| **Micro avg**                             | 0.91      | 0.73   | **0.81** | 3588    |
| **Macro avg**                             | 0.80      | 0.60   | **0.67** | 3588    |
| **Weighted avg**                          | 0.90      | 0.73   | **0.80** | 3588    |
| **Samples avg**                           | 0.83      | 0.78   | **0.79** | 3588


### Results — 20-epoch ResNet18 (GPU, #18)
### Alterations: Add dropout layer with 0.3 probability.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima*                  | 0.83      | 0.79   | **0.81** | 887     |
| *Chorthippus brunneus brunneus*           | 0.87      | 0.32   | **0.46** | 41      |
| *Pholidoptera griseoaptera*               | 0.97      | 0.76   | **0.85** | 509     |
| *Leptophyes punctatissima*                | 0.91      | 0.90   | **0.91** | 238     |
| *Gryllus campestris*                      | 0.99      | 0.90   | **0.94** | 861     |
| *Roeseliana roeselii*                     | 0.85      | 0.80   | **0.82** | 693     |
| *Pseudochorthippus parallelus parallelus* | 0.71      | 0.74   | **0.73** | 295     |
| *Omocestus viridulus*                     | 1.00      | 0.58   | **0.73** | 64      |
| **Micro avg**                             | 0.89      | 0.81   | **0.84** | 3588    |
| **Macro avg**                             | 0.89      | 0.72   | **0.78** | 3588    |
| **Weighted avg**                          | 0.89      | 0.81   | **0.84** | 3588    |
| **Samples avg**                           | 0.87      | 0.84   | **0.84** | 3588    |


### Results — 20-epoch ResNet18 (GPU, #19)
### Alterations: Add dropout layer with 0.5 probability.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima*                  | 0.85      | 0.77   | **0.81** | 887     |
| *Chorthippus brunneus brunneus*           | 0.87      | 0.32   | **0.46** | 41      |
| *Pholidoptera griseoaptera*               | 0.99      | 0.74   | **0.85** | 509     |
| *Leptophyes punctatissima*                | 0.82      | 0.94   | **0.88** | 238     |
| *Gryllus campestris*                      | 0.96      | 0.91   | **0.93** | 861     |
| *Roeseliana roeselii*                     | 0.88      | 0.79   | **0.83** | 693     |
| *Pseudochorthippus parallelus parallelus* | 0.82      | 0.67   | **0.74** | 295     |
| *Omocestus viridulus*                     | 0.83      | 0.59   | **0.69** | 64      |
| **Micro avg**                             | 0.89      | 0.80   | **0.84** | 3588    |
| **Macro avg**                             | 0.88      | 0.72   | **0.77** | 3588    |
| **Weighted avg**                          | 0.90      | 0.80   | **0.84** | 3588    |
| **Samples avg**                           | 0.88      | 0.84   | **0.85** | 3588

### Results — 20-epoch ResNet18 (GPU, #20)
### Alterations: Redistribute InsectSet clips to provide validation (52 clips) for Omocestus Viridulus.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima* | 0.85 | 0.75 | **0.80** | 887 |
| *Chorthippus brunneus brunneus* | 0.85 | 0.41 | **0.56** | 41 |
| *Pholidoptera griseoaptera* | 0.96 | 0.77 | **0.86** | 509 |
| *Leptophyes punctatissima* | 0.91 | 0.91 | **0.91** | 238 |
| *Gryllus campestris* | 0.97 | 0.90 | **0.94** | 861 |
| *Roeseliana roeselii* | 0.86 | 0.81 | **0.83** | 693 |
| *Pseudochorthippus parallelus parallelus* | 0.82 | 0.64 | **0.72** | 295 |
| *Omocestus viridulus* | 0.97 | 0.59 | **0.74** | 64 |
| **Micro avg** | 0.90 | 0.80 | **0.85** | 3588 |
| **Macro avg** | 0.90 | 0.72 | **0.79** | 3588 |
| **Weighted avg** | 0.90 | 0.80 | **0.84** | 3588 |
| **Samples avg** | 0.88 | 0.83 | **0.85** | 3588 |


### Results — 20-epoch ResNet18 (GPU, #21)
### Alterations: Redistribute InsectSet clips to provide validation (18 clips) for Omocestus Viridulus.

| Species | Precision | Recall | F1 | Support |
|---|---|---|---|---|
| *Tettigonia viridissima*                  | 0.87      | 0.83   | **0.85** | 887     |
| *Chorthippus brunneus brunneus*           | 0.93      | 0.32   | **0.47** | 41      |
| *Pholidoptera griseoaptera*               | 0.96      | 0.75   | **0.84** | 509     |
| *Leptophyes punctatissima*                | 0.88      | 0.90   | **0.89** | 238     |
| *Gryllus campestris*                      | 0.97      | 0.92   | **0.95** | 861     |
| *Roeseliana roeselii*                     | 0.86      | 0.77   | **0.81** | 693     |
| *Pseudochorthippus parallelus parallelus* | 0.90      | 0.63   | **0.74** | 295     |
| *Omocestus viridulus*                     | 0.93      | 0.78   | **0.85** | 64      |
| **Micro avg**                             | 0.91      | 0.81   | **0.86** | 3588    |
| **Macro avg**                             | 0.91      | 0.74   | **0.80** | 3588    |
| **Weighted avg**                          | 0.91      | 0.81   | **0.85** | 3588    |
| **Samples avg**                           | 0.89      | 0.85   | **0.85** | 3588    |